In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv('fashion-mnist_train.csv')
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0


In [4]:
from sklearn.model_selection import train_test_split
X = df.drop('label', axis=1).values
y = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [5]:
X_train = X_train/255
X_test = X_test/255

In [6]:
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.long)
y_test = torch.from_numpy(y_test).to(torch.long)

In [7]:
# Data augumentation transformations for the training dataset
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

In [8]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels, transform=None):
    self.features = features.reshape(-1, 1, 28, 28)
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    feature, label = self.features[index], self.labels[index]
    if self.transform:
      feature = self.transform(feature.squeeze(0).numpy()) #Apply transform and squeeze(0) to remove channel axis before applying transformations
    return feature, label

In [9]:
train_dataset = CustomDataset(X_train, y_train, transform=train_transform)
test_dataset = CustomDataset(X_test, y_test, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [10]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.features = nn.Sequential(
        # Conv layer 1 with 32 filters
        nn.Conv2d(num_features, 32, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),

        # Conv layer 2 with 64 filters
        nn.Conv2d(32, 64, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.classifier = nn.Sequential(
        # ANN layer
        nn.Flatten(),
        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(64, 10)
    )

  def forward(self, num_features):
    x = self.features(num_features)
    x = self.classifier(x)
    return x

In [11]:
learning_rate = 0.1
epochs = 40

In [12]:
model = Neural_Network(1)
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

for epoch in range(epochs):
  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    y_pred = model(batch_features)
    loss = loss_function(y_pred, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:0.8317152261734009
Epoch: 2, Loss:0.6583207845687866
Epoch: 3, Loss:0.5150300860404968
Epoch: 4, Loss:0.35783761739730835
Epoch: 5, Loss:0.3831223249435425
Epoch: 6, Loss:0.3280773162841797
Epoch: 7, Loss:0.6119374632835388
Epoch: 8, Loss:0.324773907661438
Epoch: 9, Loss:0.3158317506313324
Epoch: 10, Loss:0.5146465301513672
Epoch: 11, Loss:0.25026968121528625
Epoch: 12, Loss:0.24828439950942993
Epoch: 13, Loss:0.41131699085235596
Epoch: 14, Loss:0.43735912442207336
Epoch: 15, Loss:0.5328526496887207
Epoch: 16, Loss:0.7523937225341797
Epoch: 17, Loss:0.3928583860397339
Epoch: 18, Loss:0.22727754712104797
Epoch: 19, Loss:0.45828869938850403
Epoch: 20, Loss:0.27027428150177
Epoch: 21, Loss:0.47194844484329224
Epoch: 22, Loss:0.4126654267311096
Epoch: 23, Loss:0.2618294656276703
Epoch: 24, Loss:0.3971105217933655
Epoch: 25, Loss:0.5026035308837891
Epoch: 26, Loss:0.5837417840957642
Epoch: 27, Loss:0.4423818588256836
Epoch: 28, Loss:0.3003396987915039
Epoch: 29, Loss:0.416435

In [13]:
model.eval()
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    y_pred = model(batch_features)
    _, predicted = torch.max(y_pred, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
print(f'Accuracy: {correct/total}')

Accuracy: 0.8976666666666666
